# Fine-tune BERTić on SerbMR (Colab / Kaggle GPU)

This notebook runs `scripts/train.py --config configs/bertic.yaml` on a free GPU (NFR-02: < 1 hour).
It uses exactly the same code as local training, and checks that the data split is identical to the
committed one before training.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (Colab) or *Accelerator → GPU T4* (Kaggle).

Steps: settings → GPU check → clone and install → data (with split verification) → optional Hub login →
training → optional head-only truncation experiment → download results.

## 1. Settings

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<you>/serbian-sentiment.git" (leave empty to upload a zip instead)
BRANCH = "main"
PUSH_TO_HUB = False  # True: publish the model to the Hugging Face Hub (needs a write token)
HF_REPO_ID = ""  # e.g. "<your-hf-username>/bertic-serbmr-sentiment"
RUN_HEAD_EXPERIMENT = True  # also train with head-only truncation for comparison (~same time again)

## 2. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: switch the runtime to a T4 GPU first."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 3. Get the code and install it
With `REPO_URL` empty, upload a zip of the repository (without `data/`, `artifacts/`, `.venv/`) when prompted.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

ROOT = Path("/content" if Path("/content").exists() else "/kaggle/working") / "serbian-sentiment"
if not ROOT.exists():
    if REPO_URL:
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(ROOT)], check=True
        )
    else:
        from google.colab import files  # Colab only; on Kaggle add the repo as a dataset instead

        uploaded = files.upload()
        name = next(iter(uploaded))
        with zipfile.ZipFile(name) as z:
            z.extractall(ROOT.parent)
        inner = [p for p in ROOT.parent.iterdir() if p.is_dir() and (p / "pyproject.toml").exists()]
        if inner and inner[0] != ROOT:
            inner[0].rename(ROOT)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[transformer]"], check=True)
print("Working directory:", Path.cwd())

## 4. Data

The raw file is downloaded from the official SerbMR release (license: CC BY-NC-SA 4.0, research use).
The committed `reports/data_quality.md` records the raw file's SHA-256 and a fingerprint of the split,
and both are checked here, so the model is trained and validated on exactly the same reviews as the
local models.

In [ ]:
import re
import urllib.request

report = Path("reports/data_quality.md").read_text(encoding="utf-8")
expected_sha = re.search(r"Raw file SHA-256 \| `([0-9a-f]+)`", report).group(1)
expected_fp = re.search(r"Split fingerprint \| `([0-9a-f]+)`", report).group(1)

raw = Path("data/raw/SerbMR-2C.csv")
raw.parent.mkdir(parents=True, exist_ok=True)
if not raw.exists():
    urllib.request.urlretrieve(
        "https://github.com/vukbatanovic/SerbMR/releases/download/v1.0/SerbMR-2C.csv", raw
    )

subprocess.run(
    [sys.executable, "scripts/prepare_data.py", "--config", "configs/data.yaml"], check=True
)
new_report = Path("reports/data_quality.md").read_text(encoding="utf-8")
got_sha = re.search(r"Raw file SHA-256 \| `([0-9a-f]+)`", new_report).group(1)
got_fp = re.search(r"Split fingerprint \| `([0-9a-f]+)`", new_report).group(1)
assert got_sha == expected_sha, f"Raw file differs: {got_sha} != {expected_sha}"
assert got_fp == expected_fp, f"Split differs: {got_fp} != {expected_fp}"
print("Raw file and split match the committed report:", got_fp)

## 5. Hugging Face login (only if publishing)
Use a token with *write* access from https://huggingface.co/settings/tokens.

In [ ]:
if PUSH_TO_HUB:
    assert HF_REPO_ID, "Set HF_REPO_ID, e.g. '<your-hf-username>/bertic-serbmr-sentiment'"
    from huggingface_hub import notebook_login

    notebook_login()

## 6. Fine-tune BERTić (head+tail truncation, the main model)

In [ ]:
cmd = [
    sys.executable,
    "scripts/train.py",
    "--config",
    "configs/bertic.yaml",
    "--run-name",
    "bertic",
]
if PUSH_TO_HUB:
    cmd += ["--set", "params.push_to_hub=true", "--set", f"params.hub_model_id={HF_REPO_ID}"]
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Image, Markdown, display

display(
    Markdown(
        Path("reports/bertic/training.md")
        .read_text(encoding="utf-8")
        .split("## Learning curves")[0]
    )
)
display(Image("reports/bertic/learning_curves.png"))

## 7. Optional experiment: head-only truncation
The same model, keeping only the first 254 subword tokens. For the BiLSTM, head+tail beat head-only by 7.8 F1 points; this checks whether that holds for BERTić. The result is written to `reports/bertic_head/` and is not published.

In [ ]:
if RUN_HEAD_EXPERIMENT:
    subprocess.run(
        [
            sys.executable,
            "scripts/train.py",
            "--config",
            "configs/bertic.yaml",
            "--set",
            "params.truncation=head",
            "--set",
            "output_dir=artifacts/bertic_head",
            "--report-dir",
            "reports/bertic_head",
            "--run-name",
            "exp-bertic-head",
        ],
        check=True,
    )
    display(
        Markdown(
            Path("reports/bertic_head/training.md")
            .read_text(encoding="utf-8")
            .split("## Learning curves")[0]
        )
    )

## 8. Download the results

The zip contains `reports/bertic*/` (report, learning curves, `history.csv`, `summary.json`).
Unzip it in the repository root on your machine, then run:

```bash
python scripts/import_run.py --report-dir reports/bertic --run-name bertic-colab
python scripts/import_run.py --report-dir reports/bertic_head --run-name exp-bertic-head-colab
```

This logs the Colab runs into your local MLflow. The model weights come from the Hugging Face Hub
(or download `artifacts/bertic/model/` from this runtime if you did not publish).

In [ ]:
import shutil

bundle = shutil.make_archive("bertic_results", "zip", root_dir=".", base_dir="reports")
print("Created", bundle)
try:
    from google.colab import files

    files.download(bundle)
except ImportError:
    print("Not on Colab: download", bundle, "from the output panel.")